## **Project 2 - Alex Eyre**
### In vivo CRISPR screens identify the E3 ligase Cop1 as a modulator of macrophage infiltration and cancer immunotherapy target
### (Wang et al., Cell 2021)
=============================================================================

In [26]:
# =============================================================================
# Import libraries
# =============================================================================

import os
import sys
import logging
import numpy as np
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
from pathlib import Path

import mageck
sys.path.insert(0, "/mnt/c/Users/aweyr/OneDrive/Documents/Programs/DrugZ")
import drugz as dz

In [27]:
# =============================================================================
# Startup R
# =============================================================================

logging.getLogger("rpy2").setLevel(logging.CRITICAL)
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [28]:
%%R
# =============================================================================
# Load R Packages
# =============================================================================

suppressPackageStartupMessages({library("DESeq2") # DESeq2: 1.50.2
                               library("clusterProfiler") # clusterProfiler: 4.18.4
                               library("ComplexHeatmap") # ComplexHeatmap: 2.26.1
                               library("enrichplot") # enrichplot: 1.30.5 
                               library("org.Mm.eg.db") # org.Mm.eg.db: 1.30.5 
                               library("CellChat")}) # CellChat

=============================================================================
#### Functions
=============================================================================

In [37]:
def subset_screen(
    df: pd.DataFrame,
    start_col: int,
    end_col: int,
    *,
    id_col: int = 0,
    reset_index: bool = True,
) -> pd.DataFrame:
    
    """
    Extract a screening comparison from a published supplemental table.

    The published screening tables contain multiple comparisons arranged
    side-by-side with the first row containing the column names for each
    comparison. This function extracts one comparison, assigns the proper
    column names, removes the duplicated header row, and optionally resets
    the index.

    Parameters
    ----------
    df
        Original supplemental screening table.

    start_col
        First comparison column (inclusive).

    end_col
        Last comparison column (exclusive).

    id_col
        Column containing the gene identifier. Defaults to column 0.

    reset_index
        Whether to reset the row index after removing the header row.

    Returns
    -------
    pd.DataFrame
        Cleaned subset corresponding to one screening comparison.
    """

    subset = df.iloc[:, [id_col] + list(range(start_col, end_col))].copy()

    subset.columns = subset.iloc[0]
    subset = subset.iloc[1:]

    if reset_index:
        subset = subset.reset_index(drop=True)

    return subset

=============================================================================
#### In Vivo screens with the MusCK library uncover classic and novel regulators of immune evasion.
=============================================================================

In [41]:
# =============================================================================
# Define project paths
# =============================================================================

project_dir = Path.cwd().parents[0]
data_dir = project_dir / "data" / "supplemental"

In [42]:
# =============================================================================
# Load sgRNA Library Design
# =============================================================================

MusCK_library_design = pd.read_csv(data_dir / "Table_S1_MusCK_library_design.txt", sep = "\t")

#### MusCK Description
```
sgID: unique identifiers for each sgRNA
seq: sgRNA sequence
geneID: non-unique gene identifier, each gene contains 5 sgRNAs
```

In [22]:
# =============================================================================
# Load published supplemental tables
# =============================================================================
#
# These data are MAGeCK output from the CRISPR studies performed

# Experiment 1 - In Vitro
MusCK_invitro = pd.read_csv(data_dir / "Table_S2_In_vitro_MusCK_library_screening_on_4T1_cells.txt", sep = "\t", header = 0)

# Experiment 1 - In Vivo
MusCK_invivo = pd.read_csv(data_dir / "Table_S3_In_vivo_MusCK_library_screening_on_4T1_cells.txt", sep = "\t", header = 0)
MusCK_BvBFoxn1 = subset_screen(MusCK_invivo, 1, 14)
MusCK_BOvB      = subset_screen(MusCK_invivo, 15, 29)

In [13]:
# Summary (Experiment 1 - MusCK Library)
# In Vitro: 4T1 cells
MusCK_invitro.describe()

,num,neg|score,neg|p-value,neg|fdr,neg|rank,neg|goodsgrna,neg|lfc,pos|score,pos|p-value,pos|fdr,pos|rank,pos|goodsgrna,pos|lfc
count,4721.000000,4.721000e+03,4721.000000,4721.000000,4721.000000,4721.000000,4721.000000,4.721000e+03,4721.000000,4721.000000,4721.000000,4721.000000,4721.000000
mean,4.998729,5.404764e-01,0.609285,0.853753,2418.469180,1.777801,-0.216388,4.249298e-01,0.516839,0.903303,2371.393984,1.900445,-0.216388
std,0.035631,3.539321e-01,0.333280,0.325152,1369.323746,1.546724,0.988153,3.620037e-01,0.332265,0.156922,1372.481899,1.354622,0.988153
min,4.000000,6.390000e-09,0.000001,0.000108,17.000000,0.000000,-7.786600,7.900000e-13,0.000001,0.000825,1.000000,0.000000,-7.786600
25%,5.000000,2.009100e-01,0.377180,0.999999,1234.000000,1.000000,-0.322900,8.487900e-02,0.208440,0.843446,1183.000000,1.000000,-0.322900
50%,5.000000,5.793500e-01,0.719240,0.999999,2420.000000,1.000000,0.066535,3.067300e-01,0.500070,0.999999,2367.000000,2.000000,0.066535
75%,5.000000,8.832700e-01,0.885580,0.999999,3604.000000,3.000000,0.341160,8.045000e-01,0.831230,0.999999,3559.000000,3.000000,0.341160
max,5.000000,1.000000e+00,1.000000,0.999999,4787.000000,5.000000,2.666900,1.000000e+00,1.000000,0.999999,4774.000000,5.000000,2.666900


In [11]:
# Summary (Experiment 1 - MusCK Library)
# In Vivo: BALB/c vs. BALB/c Fox1 nu/nu
MusCK_BvBFoxn1.describe()

,id,num,neg|score,neg|p-value,neg|fdr,neg|rank,neg|goodsgrna,neg|lfc,pos|score,pos|p-value,pos|fdr,pos|rank,pos|goodsgrna,pos|lfc
count,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776
unique,4776,11,4386,4379,393,4776,11,2387,4533,4525,2,4776,5,2387
top,Trim24,5,0.1429,0.33485,0.999773,1,2,-0.58496,0.56331,0.56392,0.99997,4774,0,-0.58496
freq,1,4034,8,8,2421,1,1534,409,73,73,4772,1,4554,410


In [12]:
# Summary (Experiment 1 - MusCK Library)
# In Vivo: BALB/c +OVA vs. BALB/c
MusCK_BOvB.describe()

,id,id,num,neg|score,neg|p-value,neg|fdr,neg|rank,neg|goodsgrna,neg|lfc,pos|score,pos|p-value,pos|fdr,pos|rank,pos|goodsgrna,pos|lfc
count,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776
unique,4776,4776,11,4425,4402,363,4776,10,2722,4561,4566,13,4776,2,2721
top,Trim24,TTK,5,0.13604,0.3226,0.999814,1,2,-0.58496,0.79489,0.79475,0.999933,3966,0,-0.58496
freq,1,1,4055,7,7,2914,1,1536,476,56,56,4735,1,4775,477


#### MusCK Statistics Description

```
id: geneID
num: number of sgRNAs representing the gene
neg|score: gene-level p-value for negative selection (depletion), smaller values indicate stronger evidence that disruption of this gene decreases cell fitness under the tested condition
neg|p-value: one-sided p-value testing whether sgRNAs targeting the gene are significantly depleted relative to the background distribution
neg|fdr: Benjamini-Hochberg false discovery rate (FDR) adjusted p-value for negative selection. Controls for multiple hypothesis testing
neg|rank: rank of the gene among all genes for negative selection (1 = strongest depletion)
neg|goodsgrna: number of sgRNAs for the gene whose log-fold changes contributed to the negative-selection score after DrugZ filtering and directional consistency checks
neg|lfc: mean log₂ fold change (LFC) of sgRNAs targeting the gene, more negative values indicate stronger depletion
pos|score: gene-level p-value for positive selection (enrichment),s maller values indicate stronger evidence that disruption of this gene decreases cell fitness under the tested condition
pos|p-value:  one-sided p-value testing whether sgRNAs targeting the gene are significantly enriched relative to the background distribution
pos|fdr: Benjamini-Hochberg false discovery rate (FDR) adjusted p-value for positive selection
pos|rank: rank of the gene among all genes for positive selection (1 = strongest enrichment)
pos|goodsgrna: number of sgRNAs contributing to the positive-selection score after DrugZ filtering and directional consistency checks
pos|lfc: mean log₂ fold change (LFC) of sgRNAs targeting the gene. More positive values indicate stronger enrichment
```

=============================================================================
#### Second-round MusCK screens identify Cop1 as a regulator of TNBC progression
=============================================================================

In [ ]:
# =============================================================================
# Load published supplemental tables
# =============================================================================

# Experiment 2 - In Vivo
MusCK2_invivo = pd.read_csv(data_dir / "Table_S4_In_vivo_MusCK2_0_library_screening_on_cancer_cells.txt", sep = "\t", header = 0)
MusCK2_BvBFoxn1 = subset_screen(MusCK_invivo, 1, 14)
MusCK2_BOvBFoxn1 = subset_screen(MusCK_invivo, 15, 29)
MusCK2_BOaPD1vBFoxn1 = subset_screen(MusCK_invivo, 30, 44)

In [16]:
# Summary (Experiment 2 - MusCK 2.0 Library)
# In Vivo: BALB/c vs. BALB/c Fox1 nu/nu
MusCK2_BvBFoxn1.describe()

,id,num,neg|score,neg|p-value,neg|fdr,neg|rank,neg|goodsgrna,neg|lfc,pos|score,pos|p-value,pos|fdr,pos|rank,pos|goodsgrna,pos|lfc
count,83,83,83,83,83,83,83,83,83,83,83,83,83,83
unique,83,5,83,81,17,83,9,83,80,80,9,83,6,83
top,RFWD2,8,1.46E-12,5.00E-06,0.999905,1,1,-2.7634,1,1,0.999995,83,0,-2.7634
freq,1,78,1,3,65,1,20,1,4,4,52,1,50,1


In [23]:
# Summary (Experiment 2 - MusCK 2.0 Library)
# In Vivo: BALB/c + OVA vs. BALB/c Fox1 nu/nu
MusCK2_BOvBFoxn1.describe()

,id,id,num,neg|score,neg|p-value,neg|fdr,neg|rank,neg|goodsgrna,neg|lfc,pos|score,pos|p-value,pos|fdr,pos|rank,pos|goodsgrna,pos|lfc
count,83,83,83,83,83,83,83,83,83,83,83,83,83,83,83
unique,83,83,5,83,81,16,83,9,83,83,83,13,83,5,83
top,RFWD2,RFWD2,8,8.26E-14,5.00E-06,0.999585,1,0,-3.3159,1,1,0.999995,83,0,-3.3159
freq,1,1,78,1,3,63,1,23,1,1,1,60,1,52,1


In [24]:
# Summary (Experiment 2 - MusCK 2.0 Library)
# In Vivo: BALB/c +OVA + anti-PD-1 vs. BALB/c Foxn1 nu/nu
MusCK2_BOaPD1vBFoxn1.describe()

,id,id,num,neg|score,neg|p-value,neg|fdr,neg|rank,neg|goodsgrna,neg|lfc,pos|score,pos|p-value,pos|fdr,pos|rank,pos|goodsgrna,pos|lfc
count,83,83,83,83,83,83,83,83,83,83,83,83,83,83,83
unique,83,83,5,83,83,20,83,10,83,82,82,7,83,8,83
top,RFWD2,RFWD2,8,2.01E-14,5.00E-06,0.997178,1,1,-3.6254,1,1,0.999995,83,0,-3.6254
freq,1,1,78,1,1,59,1,20,1,2,2,71,1,34,1
